In [47]:
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount("/content/gdrive")

%cd "/content/gdrive/MyDrive/datasets"

# columns has the name of each column.
df_raw = pd.read_csv('Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv', delimiter=',', encoding='latin1')
df_raw.head()

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/MyDrive/datasets


,transaction_id,user_id,age,gender,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,academic_work_impact,addiction_level,addicted_label
0,TXN00001,U00001,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,Medium,Yes,NaN,0
1,TXN00002,U00002,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,Medium,Yes,NaN,0
2,TXN00003,U00003,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,High,No,Mild,0
3,TXN00004,U00004,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,High,Yes,Moderate,1
4,TXN00005,U00005,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Low,No,Severe,1


In [48]:
df = df_raw.copy()

df.drop(columns=["transaction_id"], inplace=True)
df.drop(columns=["user_id"], inplace=True)
df.drop(columns=["addiction_level"], inplace=True)

#df["addiction_level"].fillna("None", inplace=True)

df['academic_work_impact'] = df['academic_work_impact'].map({'Yes': 1, 'No': 0})

df = pd.get_dummies(df, columns=['gender', 'stress_level'], drop_first=False)

df.head()

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,academic_work_impact,addicted_label,gender_Female,gender_Male,gender_Other,stress_level_High,stress_level_Low,stress_level_Medium
0,21,3.23,2.01,0.89,4.55,7.55,248,154,3.95,1,0,False,True,False,False,False,True
1,24,5.09,3.81,2.24,4.44,7.66,127,71,6.71,1,0,False,False,True,False,False,True
2,31,6.06,1.36,3.83,2.35,4.92,44,106,8.68,0,0,False,False,True,True,False,False
3,32,7.83,5.85,1.51,3.54,8.23,178,107,9.77,1,1,False,False,True,True,False,False
4,25,9.96,5.92,3.42,5.27,6.21,136,177,12.55,0,1,False,True,False,False,True,False


In [49]:
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)

df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)

X_train = df_train.drop(columns=["addicted_label"])
y_train = df_train["addicted_label"]

X_val = df_val.drop(columns=["addicted_label"])
y_val = df_val["addicted_label"]

X_test = df_test.drop(columns=["addicted_label"])
y_test = df_test["addicted_label"]

In [50]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

n_features = X_train.shape[1]

model = Sequential([
    Input(shape=(n_features,)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 417 (1.63 KB)

 Trainable params: 417 (1.63 KB)

 Non-trainable params: 0 (0.00 B)

In [51]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [52]:
print(X_train.shape)
print(X_train.dtypes)
print(y_train.unique())

(6000, 16)
age                          int64
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day        int64
app_opens_per_day            int64
weekend_screen_time        float64
academic_work_impact         int64
gender_Female                 bool
gender_Male                   bool
gender_Other                  bool
stress_level_High             bool
stress_level_Low              bool
stress_level_Medium           bool
dtype: object
[0 1]


In [53]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6868 - loss: 1.9999 - val_accuracy: 0.6733 - val_loss: 0.6099
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7872 - loss: 0.4438 - val_accuracy: 0.8227 - val_loss: 0.3914
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8055 - loss: 0.4057 - val_accuracy: 0.8213 - val_loss: 0.3825
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8230 - loss: 0.3742 - val_accuracy: 0.8133 - val_loss: 0.4307
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8260 - loss: 0.3705 - val_accuracy: 0.8400 - val_loss: 0.3486
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8410 - loss: 0.3513 - val_accuracy: 0.8373 - val_loss: 0.3582
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8443 - loss: 0.3412 - val_accuracy: 0.8520 - val_loss: 0.3151
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8490 - loss: 0.3316 - val_accuracy: 0.

In [54]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [56]:
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix, accuracy_score

f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy en test:", accuracy)
print("F1-score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("\nMatriz de confusión:\n", cm)
print("\nReporte completo:\n")
print(classification_report(y_test, y_pred))

Accuracy en test: 0.8893333333333333
F1-score: 0.9213270142180094
Precision: 0.9204545454545454
Recall: 0.9222011385199241

Matriz de confusión:
 [[181  42]
 [ 41 486]]

Reporte completo:

              precision    recall  f1-score   support

           0       0.82      0.81      0.81       223
           1       0.92      0.92      0.92       527

    accuracy                           0.89       750
   macro avg       0.87      0.87      0.87       750
weighted avg       0.89      0.89      0.89       750

